# FocusTrack Component 1 - Exploratory Analysis

Context-Aware Learner Interaction State Modeling.

This notebook reads the artifacts written by `scripts/run_pipeline.py` and
explores feature distributions, discovered interaction states, transition
structure, context differences and the baseline/proposed/ablation comparison.

Run the pipeline first:

```bash
python scripts/run_pipeline.py --source csv --window 60 --states 4
```

In [ ]:
from pathlib import Path

import pandas as pd

from src.config import FEATURES_DIR, RESULTS_DIR
from src.visualization.plots import (
    plot_experiment_summary,
    plot_feature_distributions,
    plot_state_profiles,
    plot_transition_heatmap,
)

FIGURES = RESULTS_DIR / "figures"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)

## 1. Load the window-level feature table

In [ ]:
features = pd.read_csv(FEATURES_DIR / "features.csv", parse_dates=["window_start", "window_end"])
features.shape

In [ ]:
features.head()

## 2. Learning-context coverage

How many windows and sessions fall into each learning activity?

In [ ]:
features.groupby("context").agg(
    windows=("window_index", "size"),
    sessions=("session_id", "nunique"),
    participants=("participant_id", "nunique"),
)

## 3. Feature distributions

In [ ]:
plot_feature_distributions(
    features,
    ["typing_rate", "mouse_distance", "idle_ratio", "application_switch_frequency"],
    FIGURES / "feature_distributions.png",
)

## 4. Discovered interaction states

In [ ]:
profile = pd.read_csv(FEATURES_DIR / "state_profiles.csv")
plot_state_profiles(profile, FIGURES / "state_profiles.png")
profile

## 5. State transitions

`P(next state | current state)`, overall and within each learning context.

In [ ]:
transition_probs = pd.read_csv(RESULTS_DIR / "transition_probabilities.csv")
plot_transition_heatmap(transition_probs, FIGURES / "transition_heatmap.png")
transition_probs.head(10)

## 6. Does the meaning of a state depend on context?

In [ ]:
context_analysis = pd.read_csv(RESULTS_DIR / "context_effect_analysis.csv")
context_agreement = pd.read_csv(RESULTS_DIR / "context_state_agreement.csv", index_col=0)
context_analysis

In [ ]:
context_agreement

## 7. Baseline vs proposed vs ablation

In [ ]:
summary = pd.read_csv(RESULTS_DIR / "experiment_summary.csv")
plot_experiment_summary(summary, FIGURES / "experiment_summary.png")
summary